# Baseline Modeling: Ridge Regression

This notebook develops the initial predictive baseline for Rossmann Store Sales. We implement a simple linear approach using a restricted feature set (`Store`, `DayOfWeek`, `Promo`, `Open`) and track results via MLflow.

### Model Architecture:
- **Algorithm**: Ridge Regression ($\alpha=1.0$)
- **Preprocessing**: One-Hot Encoding for `DayOfWeek`.
- **Filtering**: Restricted to `Open=1` and `Sales>0` records.


In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

# Path setup
sys.path.append(str(Path.cwd().parent))
from src.train_baseline import train_baseline

# Execute automated training script
# This will generate MLflow runs and serialize the model
train_baseline()

### Model Inspection & Artifacts
Validating the serialized model and inspecting the residual distribution.


In [ ]:
# Load params and model
with open("../configs/params.yaml", "r") as f:
    params = yaml.safe_load(f)

model = joblib.load(f"../{params['model']['save_path']}")

# Load sample for prediction check
train_df = pd.read_csv(f"../{params['data']['raw_train']}", low_memory=False)
sample = train_df[train_df["Open"] == 1].sample(1000, random_state=42)

y_true = sample[params["features"]["target"]]
y_pred = model.predict(sample[params["features"]["baseline"]])

# Residue plot
residuals = y_true - y_pred
sns.histplot(residuals, bins=30, kde=True)
plt.title("Baseline Model Residuals")
plt.xlabel("Prediction Error")
plt.show()

print(f"Model pipeline artifacts verified at: {params['model']['save_path']}")